In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

In [3]:
df = pd.read_csv("../data/train.csv")

X = df.drop(columns=['target', 'ID_code'])
y = df['target']

# Feature Engineering
X['mean'] = X.mean(axis=1)
X['std'] = X.std(axis=1)
X['min'] = X.min(axis=1)
X['max'] = X.max(axis=1)
X['range'] = X['max'] - X['min']

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_26756\841639588.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['mean'] = X.mean(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_26756\841639588.py:8: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  X['std'] = X.std(axis=1)
C:\Users\Muhammad Iqbal\AppData\Local\Temp\ipykernel_26756\841639588.py:9: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining a

In [4]:
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBClassifier

# Base model (GPU)
xgb = XGBClassifier(
    scale_pos_weight=9,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    device='cuda'
)

# Parameter grid
param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.8, 1.0],
    'colsample_bytree': [0.7, 0.8, 1.0]
}

# 🔥 Use sample for faster tuning (VERY IMPORTANT)
X_sample = X_train.sample(n=50000, random_state=42)
y_sample = y_train.loc[X_sample.index]

# Random search (optimized)
random_search = RandomizedSearchCV(
    xgb,
    param_distributions=param_dist,
    n_iter=5,            # reduced
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=2             # limit CPU usage
)

# Fit on sample
random_search.fit(X_sample, y_sample)

# Best model
best_xgb = random_search.best_estimator_

print("Best Params:", random_search.best_params_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best Params: {'subsample': 1.0, 'n_estimators': 300, 'max_depth': 3, 'learning_rate': 0.05, 'colsample_bytree': 0.8}


In [5]:
# Final tuned model
final_xgb = XGBClassifier(
    n_estimators=300,
    max_depth=3,
    learning_rate=0.05,
    subsample=1.0,
    colsample_bytree=0.8,
    scale_pos_weight=9,
    random_state=42,
    eval_metric='logloss',
    tree_method='hist',
    device='cuda'
)

# Train on full training data
final_xgb.fit(X_train, y_train)

# Predict
y_prob = final_xgb.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.6).astype(int)

# Evaluate
print("F1:", f1_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

F1: 0.47762998790810157
ROC-AUC: 0.8521274464254603


d:\6th semester study\Regrerssion-Comparison\research\Lib\site-packages\xgboost\core.py:751: UserWarning: [19:53:21] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV

rf = RandomForestClassifier(random_state=42, n_jobs=-1)

param_dist_rf = {
    'n_estimators': [100, 200, 300],
    'max_depth': [5, 10, 15],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

rf_search = RandomizedSearchCV(
    rf,
    param_distributions=param_dist_rf,
    n_iter=5,
    scoring='roc_auc',
    cv=3,
    verbose=1,
    n_jobs=2
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_

print("Best RF Params:", rf_search.best_params_)

Fitting 3 folds for each of 5 candidates, totalling 15 fits
Best RF Params: {'n_estimators': 200, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_depth': 15}


In [7]:
y_prob_rf = best_rf.predict_proba(X_test)[:, 1]
y_pred_rf = (y_prob_rf >= 0.6).astype(int)

print("RF F1:", f1_score(y_test, y_pred_rf))
print("RF ROC-AUC:", roc_auc_score(y_test, y_prob_rf))

RF F1: 0.0
RF ROC-AUC: 0.8258986681379096


## 🏁 Final Model Comparison and Selection

### Models Evaluated

- Logistic Regression (with feature engineering and imbalance handling)
- Random Forest (tuned)
- XGBoost (default)
- XGBoost (tuned)

---

### Results

| Model | F1-score | ROC-AUC |
|------|----------|--------|
| Logistic Regression | ~0.47 | ~0.864 |
| Random Forest (tuned) | 0.0 | ~0.826 |
| XGBoost (default) | **~0.51** | **~0.87** |
| XGBoost (tuned) | ~0.47 | ~0.85 |

---

### Key Observations

- XGBoost achieved the best overall performance
- Random Forest failed to detect minority class (F1 = 0)
- Logistic Regression performed well but lacked complexity
- Hyperparameter tuning did not improve XGBoost performance in this case

---

### Critical Insights

- Feature engineering had the highest impact on performance
- Handling class imbalance is essential for classification tasks
- Ensemble boosting methods outperform bagging methods for this dataset
- Default model configurations can sometimes outperform tuned versions

---

### Final Model

**Selected Model: XGBoost (default configuration with GPU)**

---

### Conclusion

The final pipeline combines:

- Statistical feature engineering
- Class imbalance handling
- Threshold tuning
- GPU-accelerated XGBoost

This approach provides strong performance and serves as a solid foundation for advanced machine learning and deep learning research.